# Retrieval Validation — smarter literature retrieval

Goal: can we retrieve the papers Reactome curators cited for a gene? We build a ground-truth
set, validate the query builder, document why gene-name keyword search fails, then test a new
broad-pathway-retrieval + semantic-re-ranking pipeline.

In [1]:
import sys
sys.path.append('../reactome_llm')
sys.path.append('..')

import json
from pathlib import Path

import ReactomeNeo4jUtils as neo4jutils

## 1. Ground-truth dataset

10 randomly-sampled, human-only validation genes and their Reactome-cited PMIDs (pathway +
reaction level). The set is **locked** — the one-time generation code below is kept for
provenance only; the live path reads back the saved `../data/ground_truth_pmids.json`.

In [2]:
# ONE-TIME GENERATION -- do NOT re-run (resampling would break the locked set). Kept for provenance.
#
# validation_genes = neo4jutils.get_random_annotated_genes(n=10)          # human-only sampler
# ground_truth = {g: neo4jutils.query_literature_references_for_gene(g)    # human-only, pathway+reaction
#                 for g in validation_genes}
# Path('../data').mkdir(exist_ok=True)
# with open('../data/ground_truth_pmids.json', 'w') as f:
#     json.dump(ground_truth, f, indent=2)

In [3]:
# Locked validation set -- load ground truth from JSON (no Neo4j needed).
with open('../data/ground_truth_pmids.json') as f:
    ground_truth = {gene: [int(p) for p in pmids] for gene, pmids in json.load(f).items()}

validation_genes = list(ground_truth)
for gene, pmids in ground_truth.items():
    print(f'{gene}: {len(pmids)} PMIDs')

RASAL2: 154 PMIDs
OSBPL11: 128 PMIDs
ELOA2: 171 PMIDs
TCF7: 258 PMIDs
CD3E: 78 PMIDs
PLCB3: 97 PMIDs
CD36: 382 PMIDs
ZNF540: 21 PMIDs
MYBBP1A: 25 PMIDs
ICAM4: 19 PMIDs


## 2. Query-builder validation

`build_query_and_search_terms(gene)` returns `(search_terms, description)`: a PubMed-safe
OR-joined search string and a longer biological description for embedding. It is grounded with
Reactome pathway context when available, and falls back to LLM knowledge otherwise.

In [4]:
from QueryBuilder import build_query_and_search_terms

for gene in ['TANC1', 'ELOA2']:   # TANC1: no Reactome pathways; ELOA2: has pathway context
    pathways = neo4jutils.query_pathways_for_gene(gene)
    search_terms, description = build_query_and_search_terms(gene)
    print(f'\n=== {gene} ({len(pathways)} Reactome pathways) ===')
    print('SEARCH TERMS:', search_terms)
    print('DESCRIPTION :', description)


=== TANC1 (0 Reactome pathways) ===
SEARCH TERMS: TANC1 OR "postsynaptic density" OR "synaptic scaffolding" OR "PSD-95 interaction" OR "dendritic spine" OR "neuronal development"
DESCRIPTION : TANC1 (Tetratricopeptide Repeat, Ankyrin Repeat and Coiled-Coil Containing 1) is a postsynaptic scaffolding protein predominantly expressed in neurons, where it plays a role in the organization and maintenance of dendritic spines and excitatory synapses. It interacts with key postsynaptic density proteins including PSD-95 (via its PDZ-binding motif) and Shank family members, positioning it within the PSD-95/NMDA receptor signaling complex. TANC1 has been implicated in synaptic plasticity and has been associated with neurodevelopmental conditions, including autism spectrum disorder and intellectual disability, through its role in regulating synapse structure and function.

=== ELOA2 (38 Reactome pathways) ===
SEARCH TERMS: ELOA2 OR "Elongin A2" OR "RNA Polymerase II elongation complex" OR "HIV tr

## 3. Finding — gene-name keyword search hits a hard recall ceiling

**Setup:** narrow query (gene-name expansion) -> PubMed E-Search -> embedding re-rank.

**Result (validated across all 10 ground-truth genes):** 7/10 genes had **exactly 0** overlap
between fetched candidates (FETCH=200) and ground-truth cited PMIDs; the other 3 scored
0.008-0.053. Holds even at FETCH=500 (CD36: 6/382).

**Root cause — confirmed by sampling ground-truth abstracts directly:**
- *Dominant pattern (RASAL2, OSBPL11, ZNF540, ...):* curators cite general
  pathway/mechanism/family-level review papers as background evidence. These papers **never
  mention the specific gene by name**, so no keyword query can retrieve them.
- *Secondary pattern (CD36):* legacy / alias gene naming (GPIV, FAT, SCARB3).

Neither is fixable by better query wording — E-Search does literal keyword matching. This
motivates querying by **pathway** rather than by gene name (Section 4).

## 4. New approach — two-stage retrieval + semantic re-ranking

**Stage 1 (retrieval):** a PubMed E-Search query pulls a candidate pool.
**Stage 2 (re-rank):** embed each candidate abstract and the gene `description`, keep the top
`max_papers` by cosine similarity.

**Stage 2 is held fixed** — we compare three ways of building the *Stage-1 query*:

- **gene-name** — the bare gene symbol (baseline).
- **pathway** — the gene's Reactome pathway names (unquoted, paren-OR'd), minus mega-generic
  top-level pathways that swamp relevance ranking.
- **synonym** — the gene symbol OR'd with all UniProt protein synonyms for the gene's canonical
  protein (`build_synonym_search_query`), targeting alias-named papers (e.g. CD36 → GPIV, GP3B,
  'Fatty acid translocase'). Motivated by the CD36-style alias failures found in §3: Reactome
  reactions are defined at the protein level via UniProt, which tracks names literature uses that
  the gene symbol misses.

In [ ]:
from ReactomePubMed import ReactomePubMedRetriever
from TextEmbedder import sentence_embed, create_sentence_transformer, cosine_similarity
from QueryBuilder import (build_query_and_search_terms, build_pathway_query,
                          build_synonym_search_query, build_union_query,
                          build_partner_enrichment_query, build_retrieval_query,
                          get_reranking_target)


# ---- Stage-1 query strategies: gene -> PubMed E-Search query string ----
# Generic-pathway filtering + the paren pathway construction live in QueryBuilder, so the
# pathway component of the union query is identical to the standalone pathway-name strategy.
def genename_query(gene):
    return gene


def pathway_query(gene):
    return build_pathway_query(gene)          # generic-filtered, capped, paren-OR'd


def synonym_query(gene):
    return build_synonym_search_query(gene)   # gene symbol + all UniProt synonyms


def union_query(gene):
    return build_union_query(gene)            # gene + synonyms + pathways, combined


def partner_enrichment_query(gene):
    # pathway names from FI-partner enrichment (fetch_fis top-10, binomial+FDR) instead of
    # the direct Reactome lookup -> same _pathway_fragment construction as pathway_query,
    # but works for genes with no direct Reactome annotation. Defaults tuned via top_n x
    # fi_cutoff sweep: fi_cutoff=0.8, top_n=10, fdr_cutoff=0.05.
    return build_partner_enrichment_query(gene)


def dispatcher_query(gene):
    # Production Stage-1: union if the gene has Reactome data, else partner-enrichment.
    return build_retrieval_query(gene)


STRATEGIES = {'gene-name': genename_query, 'pathway': pathway_query, 'synonym': synonym_query}


# ---- Two-stage retrieval + re-rank (Stage 2 identical across strategies) ----
# Re-rank target: either a single `description` (legacy, gene-specific) or a list of `targets`
# (pathway-level texts). With `targets`, each candidate is scored by its MAX cosine similarity
# across the target vectors -> a paper relevant to ANY of the gene's pathways ranks high.
def retrieve_and_rerank(gene, stage1_query_fn, fetch_papers, max_papers, model,
                        description=None, targets=None, return_pool=False):
    query_text = stage1_query_fn(gene)
    if targets is None and description is None:
        _, description = build_query_and_search_terms(gene)   # legacy Stage-2 target
    retriever = ReactomePubMedRetriever(top_k_results=fetch_papers)
    retriever.maxdate = '2026/03/31'                          # match frozen MongoDB cache boundary
    docs = [d for d in retriever.lazy_load(query=query_text) if d is not None]
    pool_pmids = [int(d['uid']) for d in docs]

    target_vecs = ([sentence_embed(t, model) for t in targets] if targets
                   else [sentence_embed(description, model)])
    scored = []
    for d in docs:
        if not d.get('Summary'):
            continue
        dv = sentence_embed(d['Summary'], model)
        score = max(cosine_similarity(dv, tv) for tv in target_vecs)
        scored.append((int(d['uid']), score))
    scored.sort(key=lambda x: x[1], reverse=True)
    ranked = [pmid for pmid, _ in scored[:max_papers]]
    return (ranked, pool_pmids) if return_pool else ranked

### 4a. Stage-1 only — raw candidate-pool comparison (no re-ranking)

Isolates Stage 1: for each (gene, strategy) pair, the raw counts before any re-ranking.
Five strategies are compared — the three individual ones, their **`union`** (`build_union_query`:
gene + UniProt synonyms + generic-filtered pathway names, OR'd together), and
**`partner-enrichment`** (`build_partner_enrichment_query`: pathway names from FI-partner
enrichment — `fetch_fis` top-10 partners → binomial + FDR — instead of the direct Reactome
lookup, so it also works for genes with no existing Reactome annotation). `partner-enrichment`
is the direct comparator to `pathway`: identical query construction, pathway names from a
different source. Defaults (`fi_cutoff=0.8, top_n=10, fdr_cutoff=0.05`) were tuned via a
`top_n × fi_cutoff` sweep — `top_n=10` peaked; `fi_cutoff` was flat over 0.5–0.8 and dropped
sharply at ≥0.9 (it cuts into the top-scoring partners). Even tuned, partner-enrichment tops
out near half of direct-`pathway` recall: the same ~4 genes score 0 at every setting because
their FI partners don't enrich to the pathways whose papers curators cited.

- `GT_count` — ground-truth PMIDs for the gene
- `total_returned` — every PMID E-Search returned (**== 200 means the fetch cap was hit**, so more
  were available and a larger `FETCH_PAPERS` could help; **< 200 means E-Search was exhausted**)
- `pool_size` — real abstracts actually fetched (cache misses excluded)
- `pool_hits` — GT PMIDs found anywhere in the pool

Note: with a fixed fetch cap, the union is not guaranteed to beat the best single strategy per gene —
a very broad union (many pathways + noisy short synonyms like `FAT`) dilutes Best Match relevance and
can push good papers past the cap.

In [6]:
import pandas as pd

# Stage-1 ONLY -- no embedding, no re-ranking.
#   total_returned = every PMID esearch returned (== fetch cap => more were available)
#   pool_size      = real abstracts actually fetched (cache misses / None excluded)
#   pool_hits      = raw count of GT PMIDs found anywhere in the pool
def stage1_counts(gene, query_fn, fetch_papers=1000):
    retriever = ReactomePubMedRetriever(top_k_results=fetch_papers)
    retriever.maxdate = '2026/03/31'
    total_returned, pool = 0, []
    for d in retriever.lazy_load(query=query_fn(gene)):
        total_returned += 1
        if d is not None:
            pool.append(int(d['uid']))
    return total_returned, pool

# Stage-1 strategies: the 3 individual + 'union' + 'partner-enrichment' (pathway names from
# FI-partner enrichment instead of the direct Reactome lookup). partner-enrichment is the
# direct comparator to 'pathway': same query construction, pathway names from a different source.
STAGE1_STRATEGIES = {**STRATEGIES, 'union': union_query,
                     'partner-enrichment': partner_enrichment_query}

stage1_rows = []
for gene in validation_genes:
    G = set(ground_truth[gene])
    for strat, fn in STAGE1_STRATEGIES.items():
        total_returned, pool = stage1_counts(gene, fn)
        stage1_rows.append(dict(gene=gene, strategy=strat, GT_count=len(G),
                                total_returned=total_returned,
                                pool_size=len(pool), pool_hits=len(set(pool) & G)))

stage1_df = pd.DataFrame(stage1_rows).sort_values(['gene', 'strategy']).reset_index(drop=True)
stage1_df

,gene,strategy,GT_count,total_returned,pool_size,pool_hits
0,CD36,gene-name,382,1000,846,6
1,CD36,partner-enrichment,382,1000,989,1
2,CD36,pathway,382,1000,963,4
3,CD36,synonym,382,1000,936,4
4,CD36,union,382,1000,969,7
5,CD3E,gene-name,78,654,639,1
6,CD3E,partner-enrichment,78,1000,927,7
7,CD3E,pathway,78,1000,966,2
8,CD3E,synonym,78,1000,968,1
9,CD3E,union,78,1000,933,6


In [7]:
# Sanity check -- the synonym-expanded Stage-1 query for CD36 (the §3 alias example).
print(build_synonym_search_query('CD36'))

CD36 OR "Platelet glycoprotein 4" OR "Fatty acid translocase" OR FAT OR "Glycoprotein IIIb" OR GPIIIB OR "Leukocyte differentiation antigen CD36" OR "PAS IV" OR PAS-4 OR "Platelet collagen receptor" OR "Platelet glycoprotein IV" OR GPIV OR "Thrombospondin receptor" OR GP3B OR GP4


In [8]:
import pandas as pd


def evaluate(strategy_name, query_fn, genes, ground_truth, model,
             fetch_papers=200, max_papers=50, desc_cache=None):
    """Run one Stage-1 strategy across all genes; return a per-gene precision/recall/F1 table."""
    rows = []
    for gene in genes:
        desc = desc_cache.get(gene) if desc_cache else None
        ranked, pool = retrieve_and_rerank(gene, query_fn, fetch_papers, max_papers, model,
                                           description=desc, return_pool=True)
        G = set(ground_truth[gene])
        pool_hit, top_hit = len(set(pool) & G), len(set(ranked) & G)
        prec = top_hit / max(len(ranked), 1)
        rec = top_hit / len(G)
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        rows.append(dict(strategy=strategy_name, gene=gene, GT=len(G), pool=len(pool),
                         pool_recall=pool_hit / len(G), recall=rec, precision=prec, f1=f1))
    return pd.DataFrame(rows)


FETCH_PAPERS, MAX_PAPERS = 200, 50
model = create_sentence_transformer()

# Cache the (LLM-generated) Stage-2 description once per gene; reuse across all three strategies.
desc_cache = {g: build_query_and_search_terms(g)[1] for g in validation_genes}

results = pd.concat(
    [evaluate(name, fn, validation_genes, ground_truth, model, FETCH_PAPERS, MAX_PAPERS, desc_cache)
     for name, fn in STRATEGIES.items()],
    ignore_index=True,
)

# Per-gene detail for the new synonym strategy
results[results.strategy == 'synonym'].set_index('gene')[
    ['GT', 'pool', 'pool_recall', 'recall', 'precision', 'f1']].round(3)

2026-07-15 11:58:44,423 -  TextEmbedder - INFO - Create a new SentenceTransformer object...
2026-07-15 11:58:44,453 -  sentence_transformers.base.model - INFO - No device provided, using mps
2026-07-15 11:58:44,731 -  sentence_transformers.base.model - INFO - Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
2026-07-15 11:58:53,999 -  httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-15 11:58:59,017 -  httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-15 11:59:04,408 -  httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-15 11:59:10,381 -  httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-15 11:59:16,261 -  httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-15 11:59:24,823 -  httpx - INFO - HTTP Request: POST https://api.ant

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,GT,pool,pool_recall,recall,precision,f1
gene,,,,,,
RASAL2,154,129,0.000,0.000,0.00,0.000
OSBPL11,128,43,0.000,0.000,0.00,0.000
ELOA2,171,7,0.000,0.000,0.00,0.000
TCF7,258,189,0.008,0.008,0.04,0.013
CD3E,78,191,0.000,0.000,0.00,0.000
PLCB3,97,199,0.041,0.021,0.04,0.027
CD36,382,192,0.003,0.000,0.00,0.000
ZNF540,21,53,0.000,0.000,0.00,0.000
MYBBP1A,25,181,0.000,0.000,0.00,0.000


In [9]:
# Head-to-head comparison, micro-averaged over all ground-truth PMIDs.
rows = []
for name in STRATEGIES:
    df = results[results.strategy == name]
    gt = df['GT'].sum()
    rows.append(dict(
        strategy=name,
        genes_with_pool_hit=int((df.pool_recall > 0).sum()),
        pool_recall=(df.pool_recall * df.GT).sum() / gt,
        final_recall=(df.recall * df.GT).sum() / gt,
        mean_precision_at50=df.precision.mean(),
        mean_f1=df.f1.mean(),
    ))
summary = pd.DataFrame(rows).set_index('strategy').round(4)
summary

,genes_with_pool_hit,pool_recall,final_recall,mean_precision_at50,mean_f1
strategy,,,,,
gene-name,2,0.0015,0.0015,0.004,0.0035
pathway,7,0.0278,0.0098,0.026,0.0150
synonym,3,0.0053,0.0030,0.008,0.0040


### Results & interpretation — full pipeline (Stage 1 + re-rank to top 50)

Micro-averaged over all 1,333 ground-truth PMIDs (three individual strategies; see the summary
table above for exact values):

- **pathway** is the strongest Stage-1 signal (highest pool and final recall, non-zero for most genes).
- **synonym** helps only the alias cases (e.g. CD36, PLCB3) and stays near the gene-name baseline overall.
- The **re-ranker is the weak link**: it demotes the general/background papers curators cite, so final
  recall is well below pool recall for every strategy.

The Stage-1-only table (§4a) isolates retrieval from re-ranking and is the cleaner basis for comparing
which query pulls in the most ground-truth papers before any filtering.

### 4b. Stage-2 re-ranking target: gene description vs pathway summaries

The re-ranker embeds a comparison target and keeps the top-50 pool papers by cosine similarity.
The **gene-specific description** (from `build_query_and_search_terms`) mis-ranks the
pathway/mechanism-level papers that dominate ground truth, collapsing final recall well below
pool recall. Fix: re-rank against **pathway-level text** — `get_reranking_target(gene)` returns
the gene's pathway summaries (or, for cold-start genes, the primary `suggest_pathway_placement`
pathway), and each paper is scored by its **MAX cosine** across those per-pathway vectors.

Stage 1 is held fixed at the production dispatcher (`build_retrieval_query` = union for these
have-data genes). We compare the two re-rank targets on the same pool.

**Result (10 genes, FETCH=200, top-50):** retention (final ÷ pool recall) rose **35.9% → 53.8%**;
final recall 1.05% → 1.58%; precision@50 2.80% → 4.20%. Absolute recall stays bounded by the
~2.9% Stage-1 ceiling — this fix stops the re-ranker discarding correctly-found papers, it
doesn't find more. (All 10 validation genes have Reactome data, so the cold-start placement
branch of `get_reranking_target` is not exercised here.)

In [10]:
# Stage-2 target comparison: same dispatcher Stage-1 pool, re-ranked two ways.
#   OLD = gene-specific description (single vector)   NEW = pathway summaries (max cosine)
rerank_rows = []
for gene in validation_genes:
    G = set(ground_truth[gene])
    old_ranked, pool = retrieve_and_rerank(gene, dispatcher_query, FETCH_PAPERS, MAX_PAPERS,
                                            model, description=desc_cache[gene], return_pool=True)
    new_ranked, _ = retrieve_and_rerank(gene, dispatcher_query, FETCH_PAPERS, MAX_PAPERS,
                                        model, targets=get_reranking_target(gene), return_pool=True)
    Gp = len(set(pool) & G)
    rerank_rows.append(dict(gene=gene, GT=len(G), pool_hit=Gp,
                            pool_recall=Gp/len(G),
                            old_recall=len(set(old_ranked) & G)/len(G),
                            new_recall=len(set(new_ranked) & G)/len(G)))

rerank_df = pd.DataFrame(rerank_rows)
gt = rerank_df.GT.sum()
micro = lambda c: (rerank_df[c] * rerank_df.GT).sum() / gt
print(f"pool recall              : {micro('pool_recall'):.4f}")
print(f"final recall OLD (desc)  : {micro('old_recall'):.4f}  (retention {micro('old_recall')/micro('pool_recall'):.1%})")
print(f"final recall NEW (pwy)   : {micro('new_recall'):.4f}  (retention {micro('new_recall')/micro('pool_recall'):.1%})")
rerank_df.round(4)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

pool recall              : 0.0315
final recall OLD (desc)  : 0.0083  (retention 26.2%)
final recall NEW (pwy)   : 0.0165  (retention 52.4%)


,gene,GT,pool_hit,pool_recall,old_recall,new_recall
0,RASAL2,154,15,0.0974,0.0390,0.0519
1,OSBPL11,128,16,0.1250,0.0234,0.0625
2,ELOA2,171,1,0.0058,0.0058,0.0058
3,TCF7,258,3,0.0116,0.0000,0.0000
4,CD3E,78,4,0.0513,0.0000,0.0513
5,PLCB3,97,1,0.0103,0.0000,0.0000
6,CD36,382,0,0.0000,0.0000,0.0000
7,ZNF540,21,0,0.0000,0.0000,0.0000
8,MYBBP1A,25,1,0.0400,0.0400,0.0400
9,ICAM4,19,1,0.0526,0.0000,0.0000


## 5. Partner names in the query + entity-mention re-rank bonus

Two changes, tested together and factored apart, on the same 10 locked have-data genes.

**Change 1 (Stage 1):** `build_union_query` now OR's in the gene's top-N FI interaction-partner
names (`select_top_partner_names` → `_or_join_terms`, same quoting/dedup as gene/synonyms)
alongside the gene symbol + UniProt synonyms and the direct Reactome pathway names. The
cold-start `build_partner_enrichment_query` gets the same gene + synonyms + partner ingredients,
differing only in its pathway source (partner **enrichment** vs. direct lookup) — but that branch
isn't exercised here since all 10 validation genes have Reactome data.

**Change 2 (Stage 2):** an **additive** entity-mention bonus on top of the pathway-summary cosine
score (§4b). A paper earns `+BONUS` if its abstract mentions the gene, a UniProt synonym, or a
top-N partner (`build_entity_mention_set`). It is **never a filter** — papers naming no entity keep
their cosine rank, preserving the §3 finding that curators cite mechanism-level background papers
that name no specific entity.

Entity detection is **hybrid** (the cost-controlled option): a free case-insensitive
word-boundary string match first, then a single batched LLM pass over *only* the string-match
misses (chunked, ≈⌈misses/40⌉ calls per gene) to catch aliases/variants the synonym list lacks —
far cheaper than one LLM call per abstract.

Four combos are compared to attribute each change:

| combo | Stage-1 query | re-rank score |
|---|---|---|
| **OLD** | union, no partners | cosine only |
| **partners-only** | union + partner names | cosine only |
| **bonus-only** | union, no partners | cosine + entity bonus |
| **NEW** | union + partner names | cosine + entity bonus |

Reported per-gene and micro-averaged at **top-5** (production `max_papers`) and **top-50**
(continuity with §4a/§4b).

In [11]:
import re
from QueryBuilder import (build_entity_mention_set, select_pathway_names,
                          get_uniprot_synonyms, _or_join_terms, _pathway_fragment)
from ModelConfig import create_reactome_chat_model


# ---- Change 1 test harness: OLD vs NEW Stage-1 union query ----
# OLD = pre-change union (gene + UniProt synonyms + direct Reactome pathway names, NO partners),
#       reconstructed from the same primitives so production build_union_query stays clean.
# NEW = build_union_query, which now also OR's in top-N FI interaction-partner names.
def old_union_query(gene):
    synonyms = get_uniprot_synonyms(neo4jutils.query_accession_for_gene(gene))
    name = _or_join_terms([gene] + synonyms)
    pwy = _pathway_fragment(select_pathway_names(gene))
    return ' OR '.join(f for f in (name, pwy) if f)


def new_union_query(gene):
    return build_union_query(gene)                     # + top-N partner names OR'd in


def fetch_pool(query_text, fetch_papers=200):
    retriever = ReactomePubMedRetriever(top_k_results=fetch_papers)
    retriever.maxdate = '2026/03/31'                   # frozen MongoDB cache boundary
    return [d for d in retriever.lazy_load(query=query_text) if d is not None]


# ---- Change 2 test harness: entity-mention detection (hybrid string-match + batched LLM) ----
# Additive bonus, NEVER a filter: a paper naming no entity keeps its cosine rank (preserves the
# §3 finding that curators cite mechanism-level papers that name no specific entity).
def _word_re(entity):
    return r'(?<![A-Za-z0-9])' + re.escape(entity.lower().strip()) + r'(?![A-Za-z0-9])'


def string_mentions(text, entities):
    """Free, deterministic: case-insensitive word-boundary match of any entity in the abstract."""
    tl = text.lower()
    return any(e.strip() and re.search(_word_re(e), tl) for e in entities)


def llm_mention_pmids(entities, text_by_pmid, chunk_size=40, max_chars=1200, _stats=None):
    """Batched LLM pass over ONLY the string-match misses (one gene's worth), chunked to bound
    context -> ~ceil(len(misses)/chunk_size) calls per gene (vs one call per abstract). Returns
    the PMIDs the LLM judges to mention an entity or a clear alias/casing/spacing/morphological
    variant a plain string match would miss."""
    if not text_by_pmid:
        return set()
    model = create_reactome_chat_model()
    entity_list = ', '.join(entities)
    hits, items = set(), list(text_by_pmid.items())
    for i in range(0, len(items), chunk_size):
        chunk = items[i:i + chunk_size]
        blocks = '\n\n'.join(f'[{j}] {t[:max_chars]}' for j, (_, t) in enumerate(chunk))
        prompt = (
            'You are screening PubMed abstracts for mentions of specific biological entities.\n'
            f'ENTITIES (gene symbol, protein synonyms, interaction partners): {entity_list}\n\n'
            'An abstract is a MENTION if it names any entity OR a clear variant of one '
            '(case/spacing/hyphenation differences, morphological forms, or a well-known '
            'alias/symbol). Do NOT count mere topical relatedness with no named entity.\n\n'
            'For each numbered abstract below, respond with ONLY the bracketed numbers that are '
            'MENTIONS, comma-separated (e.g. `0, 3, 7`), or the word NONE if there are none.\n\n'
            + blocks
        )
        resp = model.invoke(prompt).content
        if _stats is not None:
            _stats['calls'] += 1
        for n in re.findall(r'\d+', resp):
            idx = int(n)
            if idx < len(chunk):
                hits.add(chunk[idx][0])
    return hits

In [12]:
# Fetch OLD + NEW pools per gene, embed once, score four combos. Reuses `model` and
# FETCH_PAPERS from §4. Reported at top-5 (production max_papers) and top-50 (continuity).
K_LIST = [5, 50]
BONUS = 0.10          # additive nudge for an entity mention; MiniLM cosines here cluster ~0.1-0.4

comp_rows, mention_log = [], []
for gene in validation_genes:
    G = set(ground_truth[gene])
    targets = get_reranking_target(gene)               # pathway-summary re-rank target (§4b)
    entities = build_entity_mention_set(gene)          # gene + synonyms + top-N partners
    tvecs = [sentence_embed(t, model) for t in targets]

    pool_old = fetch_pool(old_union_query(gene), FETCH_PAPERS)   # no partner names
    pool_new = fetch_pool(new_union_query(gene), FETCH_PAPERS)   # + partner names

    # Cosine once per unique abstract across both pools (pools overlap heavily).
    docs = {int(d['uid']): d for d in (pool_old + pool_new) if d.get('Summary')}
    cos, text = {}, {}
    for pmid, d in docs.items():
        cos[pmid] = max(cosine_similarity(sentence_embed(d['Summary'], model), tv) for tv in tvecs)
        text[pmid] = d['Summary']

    # Hybrid mention map: free string-match first, batched LLM only on the misses.
    stats = {'calls': 0}
    str_hits = {p for p in docs if string_mentions(text[p], entities)}
    misses = {p: text[p] for p in docs if p not in str_hits}
    llm_hits = llm_mention_pmids(entities, misses, _stats=stats)
    mention = {p: int(p in str_hits or p in llm_hits) for p in docs}
    mention_log.append(dict(gene=gene, pool=len(docs), string_hits=len(str_hits),
                            llm_extra_hits=len(llm_hits & set(misses)), llm_calls=stats['calls']))

    pids_old = [int(d['uid']) for d in pool_old if d.get('Summary')]
    pids_new = [int(d['uid']) for d in pool_new if d.get('Summary')]

    def rank(pids, use_bonus):
        return sorted(pids, key=lambda p: cos[p] + (BONUS * mention[p] if use_bonus else 0.0),
                      reverse=True)

    combos = {                                   # (pool pmids, apply entity bonus?)
        'OLD (no-partner + cosine)':        (pids_old, False),
        'partners-only (+partner, cosine)': (pids_new, False),
        'bonus-only (no-partner + bonus)':  (pids_old, True),
        'NEW (+partner + bonus)':           (pids_new, True),
    }
    for name, (pids, use_bonus) in combos.items():
        ranked = rank(pids, use_bonus)
        pool_hit = len(set(pids) & G)
        for K in K_LIST:
            hit = len(set(ranked[:K]) & G)
            prec = hit / min(K, len(ranked)) if ranked else 0.0
            rec = hit / len(G)
            f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
            comp_rows.append(dict(gene=gene, combo=name, K=K, GT=len(G), pool=len(pids),
                                  pool_recall=pool_hit / len(G),
                                  recall=rec, precision=prec, f1=f1))

comp_df = pd.DataFrame(comp_rows)
print('LLM-cost / mention breakdown (hybrid: string-match then LLM on misses):')
pd.DataFrame(mention_log)

/opt/anaconda3/envs/paperqa/lib/python3.10/site-packages/neo4j/_sync/work/result.py:636: UserWarning: Expected a result with a single record, but found multiple.
  warn(


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-15 12:21:28,983 -  httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-15 12:21:30,998 -  httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-15 12:21:33,366 -  httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-15 12:21:42,905 -  httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-15 12:21:56,836 -  httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-15 12:22:04,716 -  httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


LLM-cost / mention breakdown (hybrid: string-match then LLM on misses):


,gene,pool,string_hits,llm_extra_hits,llm_calls
0,RASAL2,229,92,97,4
1,OSBPL11,197,2,9,5
2,ELOA2,198,3,115,5
3,TCF7,228,16,130,6
4,CD3E,338,105,233,6
5,PLCB3,202,2,81,5
6,CD36,229,87,102,4
7,ZNF540,195,0,0,5
8,MYBBP1A,197,0,0,5
9,ICAM4,229,21,128,6


In [13]:
# Aggregate, micro-averaged over all ground-truth PMIDs, per combo at each K.
from IPython.display import display

COMBO_ORDER = ['OLD (no-partner + cosine)', 'partners-only (+partner, cosine)',
               'bonus-only (no-partner + bonus)', 'NEW (+partner + bonus)']

def aggregate(df):
    gt = df.GT.sum()
    return pd.Series({
        'genes_with_hit': int((df.recall > 0).sum()),
        'pool_recall':    (df.pool_recall * df.GT).sum() / gt,
        'micro_recall':   (df.recall * df.GT).sum() / gt,
        'mean_precision': df.precision.mean(),
        'mean_f1':        df.f1.mean(),
    })

for K in K_LIST:
    sub = comp_df[comp_df.K == K]
    agg = sub.groupby('combo', sort=False).apply(aggregate).reindex(COMBO_ORDER).round(4)
    print(f'\n===== top-{K} (micro-averaged) =====')
    display(agg)


===== top-5 (micro-averaged) =====


/var/folders/s8/_tttxchs1tq096xszwb177p80000gn/T/ipykernel_17937/2976153488.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  agg = sub.groupby('combo', sort=False).apply(aggregate).reindex(COMBO_ORDER).round(4)


,genes_with_hit,pool_recall,micro_recall,mean_precision,mean_f1
combo,,,,,
OLD (no-partner + cosine),1.0,0.0293,0.0015,0.04,0.0025
"partners-only (+partner, cosine)",2.0,0.0315,0.0030,0.08,0.0073
bonus-only (no-partner + bonus),0.0,0.0293,0.0000,0.00,0.0000
NEW (+partner + bonus),1.0,0.0315,0.0015,0.04,0.0048



===== top-50 (micro-averaged) =====


/var/folders/s8/_tttxchs1tq096xszwb177p80000gn/T/ipykernel_17937/2976153488.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  agg = sub.groupby('combo', sort=False).apply(aggregate).reindex(COMBO_ORDER).round(4)


,genes_with_hit,pool_recall,micro_recall,mean_precision,mean_f1
combo,,,,,
OLD (no-partner + cosine),5.0,0.0293,0.0158,0.042,0.0245
"partners-only (+partner, cosine)",5.0,0.0315,0.0165,0.044,0.0267
bonus-only (no-partner + bonus),5.0,0.0293,0.0128,0.034,0.0204
NEW (+partner + bonus),5.0,0.0315,0.0143,0.038,0.0236


In [14]:
# Per-gene recall at top-5 (production max_papers), all four combos + NEW-vs-OLD delta.
# A negative delta flags a gene that the two changes regressed.
piv = comp_df[comp_df.K == 5].pivot_table(index='gene', columns='combo', values='recall')
piv = piv.reindex(columns=COMBO_ORDER)
piv['delta_NEW_minus_OLD'] = piv['NEW (+partner + bonus)'] - piv['OLD (no-partner + cosine)']
piv.round(3)

combo,OLD (no-partner + cosine),"partners-only (+partner, cosine)",bonus-only (no-partner + bonus),NEW (+partner + bonus),delta_NEW_minus_OLD
gene,,,,,
CD36,0.000,0.000,0.0,0.000,0.000
CD3E,0.000,0.026,0.0,0.026,0.026
ELOA2,0.000,0.000,0.0,0.000,0.000
ICAM4,0.000,0.000,0.0,0.000,0.000
MYBBP1A,0.000,0.000,0.0,0.000,0.000
OSBPL11,0.000,0.000,0.0,0.000,0.000
PLCB3,0.000,0.000,0.0,0.000,0.000
RASAL2,0.013,0.013,0.0,0.000,-0.013
TCF7,0.000,0.000,0.0,0.000,0.000


### Results & interpretation

Run 2026-07-15, 10 locked have-data genes, FETCH=200, `BONUS=0.10`. Micro-averaged over all GT PMIDs:

| combo | top-5 recall | top-5 prec | top-50 recall | top-50 prec | top-50 F1 |
|---|---|---|---|---|---|
| OLD (no-partner + cosine) | 0.0015 | 0.040 | 0.0158 | 0.042 | 0.0245 |
| **partners-only (+partner, cosine)** | **0.0030** | **0.080** | **0.0165** | **0.044** | **0.0267** |
| bonus-only (no-partner + bonus) | 0.0000 | 0.000 | 0.0128 | 0.034 | 0.0204 |
| NEW (+partner + bonus) | 0.0015 | 0.040 | 0.0143 | 0.038 | 0.0236 |

Pool recall rose 0.0293 → 0.0315 with partner names.

**Change 1 (partner names) — keep.** Pool recall improved and, cosine-only, carried through to both
top-5 (recovered CD3E, 1→2 genes with a hit) and top-50 (recall + precision up). No metric regressed
when isolated. Shipped in `build_union_query` / `build_partner_enrichment_query`.

**Change 2 (entity-mention bonus) — drop.** `bonus-only` vs `OLD` knocked RASAL2 out of the top-5
(recall 0.0015 → 0.0000) and lowered top-50 recall/precision/F1; adding the bonus to `partners-only`
(→ `NEW`) erased the partner gain. Per-gene @5: NEW gains CD3E (+0.026) but loses RASAL2 (−0.013),
everything else flat.

**Why it backfired — confirms §3.** Most of the pool *does* mention an entity (mention table: CD3E
338/338, RASAL2 189/229), but curator-cited papers are disproportionately the mechanism/background
ones that *don't* name the gene. The mention signal is therefore **anti-correlated** with ground
truth, so rewarding it promotes non-GT papers and demotes the real ones. "Additive, not a filter"
doesn't rescue it — no `BONUS` weight turns it positive (a smaller weight just → cosine-only =
`partners-only`), so a weight sweep isn't worth running. The hybrid LLM matcher also ran over-liberal
(e.g. ELOA2: 3 string hits → +115 LLM), which only worsens the effect.

**Best config = `partners-only`** (Change 1 only). Caveat: absolute recall sits at the ~1–3%
structural ceiling, so the RASAL2/CD3E moves are single-paper (n=1) flips — but the direction is
consistent across top-5 and top-50 and matches the recall-ceiling finding (§3), so it's credible.

_(All 10 genes have Reactome data, so only `build_union_query` is exercised here. The cold-start
`build_partner_enrichment_query` partner-name change is validated separately in §6.)_

## 6. Cold-start branch: partner names in `build_partner_enrichment_query` (OLD vs NEW)

§5 only exercised the has-data union path. This section validates Change 1 for the **cold-start**
builder used when a gene has no Reactome annotation.

We force *already-annotated* genes through the cold-start enrichment path (bypassing
`build_retrieval_query`'s `query_pathways_for_gene` check). Enrichment uses **only** the gene's FI
partners — never its own annotation — so this is a legitimate cold-start simulation on genes whose
real curator-cited PMIDs we already have.

- **OLD** — enriched pathway names only (pre-Change-1 body).
- **NEW** — `build_partner_enrichment_query` as shipped: enriched pathways **+** gene + UniProt
  synonyms + top-N partner names OR'd in.

Re-rank target is fixed across OLD/NEW (the placement's primary+secondary pathway summaries — what
`get_reranking_target` uses for a real cold-start gene), so only the Stage-1 query differs.

Candidates (all have-data validation genes) are **gate-checked first** (`is_confident_placement`:
FDR<1e-3 & ≥2 mapped partners); only those clearing the gate are tested.

**RAB6C dilution probe:** RAB6C (a documented cold-start failure) produced a large, ~0%-gene-mention
pool where OR-ing gene terms into the query didn't help — the gene's own papers ranked below the
fetch cap. So beyond recall/precision we report pool size and **on-target %** (fraction of the pool
mentioning the gene/synonyms) for OLD vs NEW — the tell for whether the same broad-family dilution
recurs here.

In [ ]:
import ReactomeUtils as utils
from QueryBuilder import select_partner_enriched_pathway_names, build_partner_enrichment_query

COLD_CANDIDATES = ['CD3E', 'RASAL2', 'TCF7']

# Gate check: keep only confident placements (is_confident_placement -> FDR<1e-3 & >=2 mapped
# partners). We force these have-data genes through the cold-start path; enrichment uses only FI
# partners, never the gene's own annotation, so it's a legitimate cold-start simulation.
cold_genes = []
for g in COLD_CANDIDATES:
    pl = utils.suggest_pathway_placement(g)
    ok = utils.is_confident_placement(pl)
    prim = pl.get('primary') or {}
    print(f"{g:8s} gate={'PASS' if ok else 'FAIL'} "
          f"fdr={prim.get('fdr', float('nan')):.2e} "
          f"n_mapped={len(prim.get('mapped_genes', []))} primary={prim.get('pathway_name')!r}")
    if ok:
        cold_genes.append(g)

# OLD = enriched pathway names only (pre-Change-1 body); NEW = shipped build_partner_enrichment_query
# (enriched pathways + gene + UniProt synonyms + top-N partner names OR'd in).
def cold_old_query(gene):
    names = select_partner_enriched_pathway_names(gene)
    return _pathway_fragment(names) if names else gene

def cold_new_query(gene):
    return build_partner_enrichment_query(gene)

# Cold-start rerank target: placement primary+secondary pathway summaries (get_reranking_target's
# cold-start branch). Fixed across OLD/NEW so the only difference between them is the Stage-1 query.
def cold_targets(gene):
    pl = utils.suggest_pathway_placement(gene)
    names = ([pl['primary']['pathway_name']] if pl.get('primary') else []) + \
            [s['pathway_name'] for s in pl.get('secondary', [])]
    return [t for t in (neo4jutils.query_pathway_summary(n) for n in names) if t] or [gene]

In [ ]:
# Cold-start OLD vs NEW comparison. Reuses fetch_pool / string_mentions / model / FETCH_PAPERS /
# K_LIST from §4-§5. Re-rank target is fixed per gene (cold_targets), so only the query differs.
cold_rows, cold_probe = [], []
for gene in cold_genes:
    G = set(ground_truth[gene])
    tvecs = [sentence_embed(t, model) for t in cold_targets(gene)]
    gene_syn = [gene] + get_uniprot_synonyms(neo4jutils.query_accession_for_gene(gene))
    primary = (utils.suggest_pathway_placement(gene).get('primary') or {}).get('pathway_name', '')

    for variant, qfn in [('OLD (enriched pwy only)', cold_old_query),
                         ('NEW (+gene+syn+partners)', cold_new_query)]:
        docs = [d for d in fetch_pool(qfn(gene), FETCH_PAPERS) if d.get('Summary')]
        pids = [int(d['uid']) for d in docs]
        scored = sorted(
            ((int(d['uid']),
              max(cosine_similarity(sentence_embed(d['Summary'], model), tv) for tv in tvecs))
             for d in docs), key=lambda x: x[1], reverse=True)
        ranked = [p for p, _ in scored]
        pool_hit = len(set(pids) & G)
        on_target = sum(string_mentions(d['Summary'], gene_syn) for d in docs)   # gene/synonyms only
        cold_probe.append(dict(gene=gene, variant=variant, pool=len(pids), pool_hits=pool_hit,
                               gene_mentions=on_target,
                               on_target_pct=round(on_target / max(len(pids), 1), 3),
                               primary=primary[:38]))
        for K in K_LIST:
            hit = len(set(ranked[:K]) & G)
            prec = hit / min(K, len(ranked)) if ranked else 0.0
            rec = hit / len(G)
            f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
            cold_rows.append(dict(gene=gene, variant=variant, K=K, GT=len(G), pool=len(pids),
                                  pool_recall=pool_hit / len(G), recall=rec, precision=prec, f1=f1))

cold_df = pd.DataFrame(cold_rows)
cold_df.to_csv('../data/coldstart_comparison.csv', index=False)

print('=== RAB6C dilution probe: pool size, GT-in-pool, gene on-target% ===')
print(pd.DataFrame(cold_probe).to_string(index=False))
for K in K_LIST:
    print(f'\n=== top-{K}: OLD vs NEW ===')
    print(cold_df[cold_df.K == K].pivot_table(index='gene', columns='variant',
          values=['pool_recall', 'recall', 'precision', 'f1']).round(4).to_string())
cold_df.round(4)

### Results & interpretation

Run 2026-07-15, forced cold-start path, FETCH=200, cold-start rerank target (placement pathways).
Per-row detail in `../data/coldstart_comparison.csv`.

**Gate:** CD3E, RASAL2, TCF7 all clear (FDR 4.9e-13 / 7.2e-6 / 2.0e-10; 6 / 6 / 7 mapped partners).

**Dilution probe**
| gene | variant | pool | GT-in-pool | gene on-target % |
|---|---|---|---|---|
| CD3E | OLD | 197 | 0 | 0.0% |
| CD3E | **NEW** | 197 | **4** | 0.5% |
| RASAL2 | OLD | 193 | 14 | 0.0% |
| RASAL2 | **NEW** | 198 | **16** | 0.5% |
| TCF7 | OLD | 200 | 4 | 1.0% |
| TCF7 | **NEW** | 200 | 4 | 5.0% |

**Recall / precision (OLD → NEW)**
| gene | top-5 recall | top-5 prec | top-50 recall | top-50 prec |
|---|---|---|---|---|
| CD3E | 0.000 → **0.026** | 0.00 → **0.40** | 0.000 → **0.038** | 0.00 → **0.06** |
| RASAL2 | 0.0065 → 0.0065 | 0.20 → 0.20 | 0.052 → 0.046 | 0.16 → 0.14 |
| TCF7 | 0.000 → 0.000 | 0.00 → 0.00 | 0.000 → 0.000 | 0.00 → 0.00 |

**Verdict — partner names help or are neutral in the cold-start branch; keep.**
- **CD3E — clear win.** The enriched-pathway-only query ("Co-inhibition by PD-1") found **0** of CD3E's
  GT papers; adding gene+synonyms+partners surfaced **4** (top-5 recall 0 → 0.026, precision 0 → 0.40).
  The entity fragment did real work here.
- **RASAL2 — ~neutral.** GT-in-pool +2 (14 → 16) but final recall flat @5 and a hair lower @50
  (0.052 → 0.046) — a small-n reranking wobble (the added gene-mentioning papers, which are *not* the
  GT ones, reshuffle the top). Not a dilution blowup.
- **TCF7 — no change** (4 GT in pool either way; none rank into the top-50 under the placement target).

**RAB6C comparison — mechanism present, failure NOT reproduced.**
- on-target % rose with NEW for all three (0 → 0.5%, 0 → 0.5%, 1% → 5%), so OR-ing gene terms *does*
  pull some gene-mentioning papers into the pool — unlike RAB6C, where it stayed ~0%.
- But absolute on-target % is still low single digits and pool sizes stayed pinned at ~200 (the fetch
  cap): most gene papers still rank below the cap by PubMed relevance — the same RAB6C mechanism, just
  not catastrophic. Crucially `pool_hits` never *dropped* (CD3E +4, RASAL2 +2, TCF7 +0), so the union
  does not degrade cold-start retrieval the way an `[gene] AND (pathways)` form did (36 → 4).
- Pushing on-target % substantially higher needs the documented RAB6C fix — merging a separate
  gene-name esearch result *set* into the pool — which belongs in the retriever, not the query builder.

Note: RASAL2 is the closest RAB6C analog (tight RHO-GTPase family → broad "RHO GTPase cycle" pathway)
and it's exactly the ~neutral case; CD3E's tighter, more specific TCR module is where the entity terms
paid off. Small-n throughout (single-digit GT-in-pool), so read directionally.